# Notebook 06: Linear Probes and Representation Analysis

What does a model actually know at each layer? Linear probes let you answer this directly -- train a simple classifier on internal activations and see what sticks. This notebook walks you through the technique.

## Section 1: What Are Linear Probes?

A **linear probe** is a simple classifier (or regressor) trained on top of a model's internal activations. The idea: if a linear model can predict property P from layer L's activations, then information about P is **linearly accessible** at layer L.

Think of it as a diagnostic tool -- you're not modifying the model, just asking "what information is already there?"

Why linear? If information is linearly encoded, the model can easily use it (since attention and MLPs operate on linear combinations of the residual stream). Non-linear probes tell you less -- they might extract information the model itself can't access.

## Section 2: Probing for Factual Knowledge

Let's probe whether GPT-2 linearly encodes factual attributes (like "is this entity a country?") in its residual stream. Run the cell below and you'll see how probe accuracy changes layer by layer.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# Create a simple dataset: sentences about countries vs non-countries
countries = ["France", "Germany", "Japan", "Brazil", "Canada", "Italy", "Spain", "China", "India", "Mexico",
             "Australia", "Egypt", "Nigeria", "Sweden", "Poland", "Chile", "Korea", "Turkey", "Iran", "Cuba"]
non_countries = ["Apple", "Google", "Python", "Tennis", "Piano", "Coffee", "Mountain", "River", "Diamond", "Thunder",
                 "Algebra", "Guitar", "Oxygen", "Marble", "Velvet", "Compass", "Falcon", "Granite", "Whisper", "Eclipse"]

# Generate prompts: "The word {X} refers to"
prompts_pos = [f"The word {c} refers to" for c in countries]
prompts_neg = [f"The word {n} refers to" for n in non_countries]

all_prompts = prompts_pos + prompts_neg
labels = np.array([1] * len(prompts_pos) + [0] * len(prompts_neg))

# Collect activations at each layer for the last token (before "refers")
layer_activations = {l: [] for l in range(model.cfg.n_layers)}

for prompt in all_prompts:
    _, cache = model.run_with_cache(prompt)
    for l in range(model.cfg.n_layers):
        # Take the activation at the position of the entity word
        # (second-to-last position before "refers to")
        act = cache[f"blocks.{l}.hook_resid_post"][0, -3, :].detach().cpu().numpy()
        layer_activations[l].append(act)

for l in layer_activations:
    layer_activations[l] = np.array(layer_activations[l])

print(f"Collected activations for {len(all_prompts)} prompts across {model.cfg.n_layers} layers")
print(f"Activation shape per layer: {layer_activations[0].shape}")

In [ ]:
# Train a logistic regression probe at each layer
probe_accuracies = []

for layer in range(model.cfg.n_layers):
    X = layer_activations[layer]
    
    # Cross-validated accuracy
    clf = LogisticRegression(max_iter=1000, C=1.0)
    scores = cross_val_score(clf, X, labels, cv=5, scoring='accuracy')
    probe_accuracies.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), probe_accuracies, 'bo-')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Chance level')
plt.xlabel("Layer")
plt.ylabel("Probe Accuracy (5-fold CV)")
plt.title("Linear Probe: 'Is this word a country?'")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.3, 1.05)
plt.show()

print(f"Best layer: {np.argmax(probe_accuracies)} (accuracy: {max(probe_accuracies):.3f})")
print(f"Chance level: 0.50")

## Section 3: The Geometry of Truth

Here's a remarkable finding: [Marks & Tegmark (2023)](https://arxiv.org/abs/2310.06824) discovered that LLMs have a clean **linear structure** for truthfulness. True and false statements are linearly separable in activation space, and this "truth direction" is consistent across different types of facts.

This means you can:
1. Find the "truth direction" by training a probe on true/false statements
2. Project any statement onto this direction to estimate the model's confidence in its truth
3. This direction generalizes across topics -- it's a feature of the representation, not a topic-specific hack

In [ ]:
# Create true/false statement pairs
true_statements = [
    "The sky is blue",
    "Water freezes at zero degrees Celsius",
    "The Earth orbits the Sun",
    "Humans have two eyes",
    "Paris is in France",
    "Fish live in water",
    "The Moon orbits Earth",
    "Iron is a metal",
    "Penguins are birds",
    "The Pacific is an ocean",
    "Dogs are mammals",
    "The Sun is a star",
    "Snow is cold",
    "Triangles have three sides",
    "Gold is a metal",
]

false_statements = [
    "The sky is green",
    "Water freezes at fifty degrees Celsius",
    "The Sun orbits the Earth",
    "Humans have three eyes",
    "Paris is in Germany",
    "Fish live on land",
    "The Earth orbits the Moon",
    "Iron is a gas",
    "Penguins are fish",
    "The Pacific is a mountain",
    "Dogs are reptiles",
    "The Sun is a planet",
    "Snow is hot",
    "Triangles have five sides",
    "Gold is a liquid",
]

all_statements = true_statements + false_statements
truth_labels = np.array([1] * len(true_statements) + [0] * len(false_statements))

# Collect activations at the last token
truth_activations = {l: [] for l in range(model.cfg.n_layers)}

for stmt in all_statements:
    _, cache = model.run_with_cache(stmt)
    for l in range(model.cfg.n_layers):
        act = cache[f"blocks.{l}.hook_resid_post"][0, -1, :].detach().cpu().numpy()
        truth_activations[l].append(act)

for l in truth_activations:
    truth_activations[l] = np.array(truth_activations[l])

# Probe at each layer
truth_probe_accuracies = []
for layer in range(model.cfg.n_layers):
    clf = LogisticRegression(max_iter=1000, C=1.0)
    scores = cross_val_score(clf, truth_activations[layer], truth_labels, cv=5)
    truth_probe_accuracies.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), truth_probe_accuracies, 'go-', label="Truth probe")
plt.plot(range(model.cfg.n_layers), probe_accuracies, 'bo-', alpha=0.5, label="Country probe")
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Chance')
plt.xlabel("Layer")
plt.ylabel("Probe Accuracy")
plt.title("Linear Probes: Truth vs. Country Classification by Layer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.3, 1.05)
plt.show()

## Section 4: Extracting the Truth Direction

Once we know truth is linearly encoded, we can extract the actual direction vector and use it to analyze new statements.

In [ ]:
# Train probe on best layer and extract the direction
best_layer = np.argmax(truth_probe_accuracies)
print(f"Using layer {best_layer} (accuracy: {truth_probe_accuracies[best_layer]:.3f})")

X = truth_activations[best_layer]
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X, truth_labels)

# The truth direction is the weight vector of the logistic regression
truth_direction = clf.coef_[0]  # (d_model,)
truth_direction = truth_direction / np.linalg.norm(truth_direction)  # Normalize

# Project training examples onto truth direction
projections = X @ truth_direction

plt.figure(figsize=(10, 4))
plt.hist(projections[truth_labels == 1], bins=15, alpha=0.6, label="True statements", color='green')
plt.hist(projections[truth_labels == 0], bins=15, alpha=0.6, label="False statements", color='red')
plt.xlabel("Projection onto Truth Direction")
plt.ylabel("Count")
plt.title("Statements Projected onto the Learned Truth Direction")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Test on new statements
test_statements = [
    ("The moon is made of cheese", False),
    ("Elephants are large animals", True),
    ("Two plus two equals five", False),
    ("The Amazon is a river", True),
    ("Cats can fly", False),
    ("Oxygen is needed for breathing", True),
]

print("\nTesting truth direction on new statements:")
print("-" * 60)
for stmt, expected in test_statements:
    _, cache = model.run_with_cache(stmt)
    act = cache[f"blocks.{best_layer}.hook_resid_post"][0, -1, :].detach().cpu().numpy()
    proj = act @ truth_direction
    predicted = "True" if clf.predict(act.reshape(1, -1))[0] == 1 else "False"
    print(f"  '{stmt}'")
    print(f"    Expected: {expected}, Predicted: {predicted}, Projection: {proj:.3f}")

## Section 5: Contrast-Consistent Search (CCS)

[Burns et al. (2022)](https://arxiv.org/abs/2212.03827) proposed **CCS** -- an *unsupervised* method for finding truth directions. Instead of needing labeled true/false data, CCS works by:

1. Generate pairs of contradictory statements: (S, not-S)
2. Collect activations for both
3. Find a direction that:
   - Assigns opposite truth values to S and not-S (consistency)
   - Is confident (outputs near 0 or 1, not 0.5)

This is powerful because it doesn't need any labels -- it discovers truth-related features from the model's own representations.

In [ ]:
# Simplified CCS: Find direction maximizing contrast between paired statements
# CCS uses: L = consistency_loss + confidence_loss
# consistency: p(S) + p(not S) ~ 1
# confidence: p(S) is close to 0 or 1

paired_true = true_statements[:10]
paired_false = false_statements[:10]

layer = best_layer
acts_true = truth_activations[layer][:10]   # True versions
acts_false = truth_activations[layer][len(true_statements):len(true_statements)+10]  # False versions

# CCS probe: learn direction where p(true) + p(false) ~ 1 and p is confident
class CCSProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

probe = CCSProbe(768)
optimizer = torch.optim.Adam(probe.parameters(), lr=1e-2)

acts_true_t = torch.tensor(acts_true, dtype=torch.float32)
acts_false_t = torch.tensor(acts_false, dtype=torch.float32)

for step in range(500):
    p_true = probe(acts_true_t).squeeze()
    p_false = probe(acts_false_t).squeeze()
    
    # Consistency loss: p(true) + p(false) should ~ 1
    consistency_loss = ((p_true + p_false - 1) ** 2).mean()
    
    # Confidence loss: predictions should be near 0 or 1
    confidence_loss = (torch.min(p_true, 1 - p_true) ** 2).mean() + \
                      (torch.min(p_false, 1 - p_false) ** 2).mean()
    
    loss = consistency_loss + confidence_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Check results (CCS might flip true/false -- we check both orientations)
with torch.no_grad():
    p_true_final = probe(acts_true_t).squeeze().numpy()
    p_false_final = probe(acts_false_t).squeeze().numpy()

# Check if true statements get higher or lower probability
if p_true_final.mean() > p_false_final.mean():
    print("CCS correctly assigns higher probability to true statements")
else:
    print("CCS flipped orientation (common -- just flip the sign)")
    
print(f"\nTrue statement probs:  {np.round(p_true_final, 2)}")
print(f"False statement probs: {np.round(p_false_final, 2)}")
print(f"\nMean true: {p_true_final.mean():.3f}, Mean false: {p_false_final.mean():.3f}")

---
### Running Example: IOI — Probing for Name Identity

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

Here we train a linear probe to predict which name (Mary vs the alternative) the model will output, based on intermediate activations at the final token position. This reveals at which layer the model has linearly encoded "the answer is Mary" versus "the answer is not Mary".

In [ ]:
# Running Example: IOI — Probing for Name Identity
from sklearn.linear_model import LogisticRegression
import numpy as np

# Collect (activation, label) pairs from IOI variants
# label=1 if Mary is the indirect object (should be predicted)
prompts_mary = [f"When Mary and {name} went to the store, {name} gave a drink to" 
                for name in ["John", "Bob", "Alice", "Tom", "Sarah"]]
prompts_other = [f"When {name} and Mary went to the store, Mary gave a drink to"
                 for name in ["John", "Bob", "Alice", "Tom", "Sarah"]]

probe_data = []
for p in prompts_mary:
    _, c = model.run_with_cache(p)
    for l in range(model.cfg.n_layers):
        act = c[f"blocks.{l}.hook_resid_post"][0, -1].detach().cpu().numpy()
        probe_data.append((l, act, 1))

for p in prompts_other:
    _, c = model.run_with_cache(p)
    for l in range(model.cfg.n_layers):
        act = c[f"blocks.{l}.hook_resid_post"][0, -1].detach().cpu().numpy()
        probe_data.append((l, act, 0))

# Probe per layer: can we linearly predict "will the model output Mary"?
print("IOI Name Identity Probe — accuracy by layer:")
print("-" * 50)
for layer in range(model.cfg.n_layers):
    X = np.array([d[1] for d in probe_data if d[0] == layer])
    y = np.array([d[2] for d in probe_data if d[0] == layer])
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y)
    acc = clf.score(X, y)
    if acc > 0.7:
        print(f"  Layer {layer:2d}: probe accuracy = {acc:.2f}")

## Exercises

### Exercise 1: Probe for Part of Speech

Train a linear probe to classify whether the last token in a prompt is a noun, verb, or adjective. Use 200 examples per class. Which layers are most informative for part-of-speech information?

<details>
<summary>Hint</summary>

Create prompts that end in a noun (e.g., "I saw the dog"), a verb (e.g., "She likes to run"), or an adjective (e.g., "The sky is blue"). Collect residual stream activations at the last token position for each layer. Train a separate `LogisticRegression` probe per layer using sklearn, and plot accuracy vs layer number.

</details>

In [ ]:
# Exercise 1: Probe for Part of Speech

# Create prompts ending in nouns, verbs, adjectives
# TODO: Try expanding these lists for a more robust probe!
nouns = ["dog", "cat", "house", "tree", "car", "book", "river", "mountain", "table", "phone"]
verbs = ["run", "jump", "sing", "write", "eat", "sleep", "swim", "dance", "read", "think"]
adjectives = ["happy", "tall", "bright", "cold", "fast", "soft", "dark", "warm", "loud", "calm"]

noun_prompts = [f"I saw the {n}" for n in nouns]
verb_prompts = [f"She likes to {v}" for v in verbs]
adj_prompts = [f"The sky is {a}" for a in adjectives]

all_pos_prompts = noun_prompts + verb_prompts + adj_prompts
pos_labels = np.array([0]*len(noun_prompts) + [1]*len(verb_prompts) + [2]*len(adj_prompts))

# Collect residual stream activations at each layer
pos_activations = {l: [] for l in range(model.cfg.n_layers)}
for p in all_pos_prompts:
    _, cache = model.run_with_cache(p)
    for l in range(model.cfg.n_layers):
        act = cache[f"blocks.{l}.hook_resid_post"][0, -1, :].detach().cpu().numpy()
        pos_activations[l].append(act)
for l in pos_activations:
    pos_activations[l] = np.array(pos_activations[l])

# Train LogisticRegression per layer
accuracies = []
for layer in range(model.cfg.n_layers):
    clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
    scores = cross_val_score(clf, pos_activations[layer], pos_labels, cv=5)
    accuracies.append(scores.mean())

# Plot accuracy vs layer
plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), accuracies, 'mo-')
plt.axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Chance (3 classes)')
plt.xlabel("Layer")
plt.ylabel("Probe Accuracy (5-fold CV)")
plt.title("Linear Probe: Part of Speech (Noun / Verb / Adjective)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Best layer: {np.argmax(accuracies)} (accuracy: {max(accuracies):.3f})")

### Exercise 2: The Geometry of Sentiment

Find the "sentiment direction" in GPT-2's activation space. Collect activations for 100 positive and 100 negative sentiment sentences. Compute the mean difference vector. Does projecting new sentences onto this direction predict sentiment?

<details>
<summary>Hint</summary>

`sentiment_direction = mean(positive_activations) - mean(negative_activations)`. Project test activations onto this direction using a dot product and threshold at zero. Positive projections should correspond to positive sentiment and vice versa. Pick a middle-to-late layer where semantic information is well-developed.

</details>

In [ ]:
# Exercise 2: The Geometry of Sentiment

# TODO: Try adding more prompts for a more robust sentiment direction!
positive_prompts = [
    "This movie was fantastic", "I love this beautiful day",
    "The food was absolutely delicious", "What a wonderful experience",
    "I am so happy and grateful", "This is truly amazing work",
    "The performance was brilliant", "I feel great about this",
]
negative_prompts = [
    "This movie was terrible", "I hate this awful day",
    "The food was absolutely disgusting", "What a horrible experience",
    "I am so sad and miserable", "This is truly awful work",
    "The performance was dreadful", "I feel awful about this",
]

# Collect activations at a chosen layer
# TODO: Try different layers to see where sentiment is best encoded!
chosen_layer = 8
pos_acts = []
neg_acts = []
for p in positive_prompts:
    _, cache = model.run_with_cache(p)
    pos_acts.append(cache[f"blocks.{chosen_layer}.hook_resid_post"][0, -1].detach().cpu().numpy())
for p in negative_prompts:
    _, cache = model.run_with_cache(p)
    neg_acts.append(cache[f"blocks.{chosen_layer}.hook_resid_post"][0, -1].detach().cpu().numpy())
pos_acts = np.array(pos_acts)
neg_acts = np.array(neg_acts)

# Compute sentiment direction
sentiment_direction = pos_acts.mean(axis=0) - neg_acts.mean(axis=0)
sentiment_direction = sentiment_direction / np.linalg.norm(sentiment_direction)

# Visualize projections of training data
pos_projs = pos_acts @ sentiment_direction
neg_projs = neg_acts @ sentiment_direction

plt.figure(figsize=(10, 4))
plt.hist(pos_projs, bins=10, alpha=0.6, label="Positive", color='green')
plt.hist(neg_projs, bins=10, alpha=0.6, label="Negative", color='red')
plt.xlabel("Projection onto Sentiment Direction")
plt.ylabel("Count")
plt.title(f"Sentiment Direction (Layer {chosen_layer})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Test on new prompts
test_prompts = [
    ("I enjoyed every moment of it", "pos"),
    ("This was a waste of time", "neg"),
    ("The sunset was breathtaking", "pos"),
    ("I regret buying this product", "neg"),
]
print("Testing sentiment direction on new prompts:")
print("-" * 60)
for text, expected in test_prompts:
    _, cache = model.run_with_cache(text)
    act = cache[f"blocks.{chosen_layer}.hook_resid_post"][0, -1].detach().cpu().numpy()
    proj = act @ sentiment_direction
    predicted = "pos" if proj > 0 else "neg"
    match = "OK" if predicted == expected else "MISS"
    print(f"  [{match}] '{text}' -> projection={proj:.3f}, predicted={predicted}, expected={expected}")

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- Linear probes reveal what information is linearly encoded in model activations
- Truth, factual attributes, and many semantic properties are linearly separable
- The "truth direction" generalizes across topics
- CCS finds truth directions without labels (unsupervised)
- Probing accuracy varies by layer -- different layers encode different information

**Watch out for:**
- Probe accuracy doesn't mean the model *uses* that information (it might be encoded but unused)
- High probe accuracy with a complex probe doesn't mean the model has easy access to the info
- Probes can pick up on spurious correlations in the training data

**Further reading:**
- [The Geometry of Truth](https://arxiv.org/abs/2310.06824) (Marks & Tegmark, 2023)
- [Discovering Latent Knowledge (CCS)](https://arxiv.org/abs/2212.03827) (Burns et al., 2022)
- [Language Models Don't Always Say What They Think](https://arxiv.org/abs/2305.04388) (Turpin et al., 2023)

**Next**: Notebook 07 -- Representation Engineering (using probing insights to *steer* model behavior)